# 02 — 단일 회귀 (경로 C)

**하나의 회귀 모델**만 HPO. 5종 (`lgbm / xgb / catboost / et / enet`) 중 선택.

- `MODEL_NAME` 스위치로 모델 선택 → 노트북을 모델별로 재실행 (병렬 가능)
- die-level 학습 + unit-level RMSE objective
- `TARGET_TRANSFORM` 선택 가능 (`'none' | 'log1p' | 'yeo-johnson'`)
- 출력: `4_output/final/reg_only/{MODEL_NAME}/` 하위에 아티팩트 6종

**로컬 + Colab 양쪽 지원.** Colab 사용 시 `GDRIVE_FINAL_ID`만 채우면 됨.

## 1. 환경 설정 + 모듈 import

### Colab 사용 가이드
1. 로컬에서 `3_modeling/final/` 폴더를 **zip**으로 압축 (이름: `final.zip`)
2. Google Drive에 업로드
3. 공유 링크에서 파일 ID 복사 → 아래 `GDRIVE_FINAL_ID` 에 붙여넣기
4. code/data/preprocessing zip의 ID는 기존 CLAUDE.md 값 그대로 사용 (수정 불필요)

In [1]:
import os, sys

# ── Colab 사용 시에만 채울 것 (로컬은 무시됨) ──
GDRIVE_FINAL_ID = '1HR7LlQmp4n9wGh2WneyVex2mCZ-poiY9'   # ★ Colab에서 final.zip 업로드 후 공유 ID 입력

try:
    import google.colab
    from google.colab import drive
    drive.mount('/content/drive')
    # code / dataset / preprocessing (기존 CLAUDE.md ID)
    if not os.path.exists('/content/project/setup.py'):
        os.system('pip install -q gdown')
        os.system('gdown 1AD4PDBnDVjp-LSna6puB7qLnpBqB7j_I -O /content/code.zip')
        os.system('unzip -qo /content/code.zip -d /content/project')
        os.makedirs('/content/project/0_data', exist_ok=True)
        os.system('gdown 1yOUo0_wPLcuZBSJIK592b00YkSIlk4zO -O /content/project/0_data/dataset.zip')
        os.system('unzip -qo /content/project/0_data/dataset.zip -d /content/project/0_data')
        os.remove('/content/project/0_data/dataset.zip')
    if not os.path.exists('/content/project/2_preprocessing/cleaning.py'):
        os.system('gdown 1Rh0ByOS4Gama8XHuvY7KkOHo278H9YLr -O /content/preprocessing.zip')
        os.system('unzip -qo /content/preprocessing.zip -d /content/project')
    # final/ 모듈 (사용자 제공 ID)
    if not os.path.exists('/content/project/3_modeling/final/modules/hpo.py'):
        assert GDRIVE_FINAL_ID, 'GDRIVE_FINAL_ID가 비어있음 — final.zip 공유 ID를 입력하세요'
        os.makedirs('/content/project/3_modeling/final', exist_ok=True)
        os.system(f'gdown {GDRIVE_FINAL_ID} -O /content/final.zip')
        os.system('unzip -qo /content/final.zip -d /content/project/3_modeling/final')
    sys.path.insert(0, '/content/project')
    %run /content/project/setup.py
except ImportError:
    %run ../../setup.py

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

from utils.config import PROJECT_ROOT, SEED, TARGET_COL, KEY_COL, OUTPUT_DIR
from utils.data import load_all, get_feat_cols, split_xs
from utils.evaluate import rmse

MODEL_ROOT = os.path.join(PROJECT_ROOT, "3_modeling")
if MODEL_ROOT not in sys.path:
    sys.path.insert(0, MODEL_ROOT)

from final.modules import preprocess, hpo, models

print(f'PROJECT_ROOT = {PROJECT_ROOT}')
print(f'Available models: {models.AVAILABLE_MODELS}')

setup 완료
PROJECT_ROOT = c:\Users\Dell5371\Desktop\기업연계프로젝트
Available models: ['lgbm', 'xgb', 'catboost', 'et', 'enet', 'zitboost']


## 2. 실험 설정

- `MODEL_NAME`: `'lgbm' / 'xgb' / 'catboost' / 'et' / 'enet'` 중 하나. **한 번에 1개 모델 학습** — 다른 모델은 이 노트북을 다시 실행(또는 복사본 병렬 실행).
- `TARGET_TRANSFORM`: `'none' / 'log1p' / 'yeo-johnson'`. zero-inflated + skew 큰 target에 log1p 권장.
- `PARAMS`: 전처리 override. 비워두면 `preprocess.DEFAULT_PARAMS` 그대로.

In [2]:
# ── 모델 선택 (하나만) ──
MODEL_NAME = 'enet'        # ★ 'lgbm' | 'xgb' | 'catboost' | 'et' | 'enet'
assert MODEL_NAME in {'lgbm', 'xgb', 'catboost', 'et', 'enet'}, \
    f'MODEL_NAME invalid: {MODEL_NAME}'

# ── 실험 식별 ──
EXP_ID   = f'reg-{MODEL_NAME}-001'
EXP_MEMO = f'경로 C — {MODEL_NAME} 단일 회귀'
USER     = 'jh'

# ── Optuna 예산 ──
N_TRIALS = 5
N_FOLDS  = 5

# ── REUSE 모드 (기존 best_params.json 재사용 → HPO 스킵, refit만 실행) ──
# 전처리만 바꾸고 HP는 그대로 써서 "개선 여부 저비용 검증" 용도.
REUSE_BEST_PARAMS     = False   # ★ True: 기존 HP 재사용 / False: 정상 HPO
PREV_BEST_PARAMS_PATH = ''      # 로컬: best_params.json 경로
GDRIVE_PREV_PARAMS_ID = ''      # Colab: 해당 JSON을 Drive 업로드 후 공유 ID

# ── 타깃 변환 (경로 C만 선택 가능) ──
TARGET_TRANSFORM = 'log1p'   # 'none' | 'log1p' | 'yeo-johnson' — 1차 실험 best
CLIP_Y_EXTREME   = True

# ── 출력 경로 (모델별 서브폴더) ──
OUT_DIR = os.path.join(OUTPUT_DIR, 'final', 'reg_only', MODEL_NAME)
DB_PATH = os.path.join(OUT_DIR, f'optuna_{USER}_{EXP_ID}.db')
os.makedirs(OUT_DIR, exist_ok=True)

# ── 전처리 파라미터 override (transform별 1차 Optuna top-50 mode 기반) ──
# 1차 Two-Stage HPO DB(43 study, 13,800 trial) 분석으로 target_transform 별 최적 탐색.
# plan §9.5 L4: corr_keep_by='target_corr' 는 모든 transform 에서 'std' 로 강제 (leakage 회피).
PARAMS_BY_TRANSFORM = {
    'yeo-johnson': {
        # 1차 best RMSE 0.001141 — log1p/none 대비 4배 이상 우세
        'missing_threshold':          0.9,
        'corr_threshold':             0.98,
        'corr_keep_by':               'std',
        'add_indicator':              False,
        'indicator_threshold':        0.15,   # add_indicator=False 라 무영향
        'spatial_max_dist':           1.0,
        'post_impute_corr_threshold': 0.98,
        'post_impute_corr_keep_by':   'std',
    },
    'log1p': {
        # 1차 best RMSE 0.005442
        'missing_threshold':          0.5,
        'corr_threshold':             0.90,
        'corr_keep_by':               'std',
        'add_indicator':              True,
        'indicator_threshold':        0.25,
        'spatial_max_dist':           5.0,
        'post_impute_corr_threshold': 0.99,
        'post_impute_corr_keep_by':   'std',
    },
    'none': {
        # 1차 best RMSE 0.005516
        'missing_threshold':          0.4,
        'corr_threshold':             0.90,
        'corr_keep_by':               'std',
        'add_indicator':              True,
        'indicator_threshold':        0.05,
        'spatial_max_dist':           5.0,
        'post_impute_corr_threshold': 0.98,
        'post_impute_corr_keep_by':   'std',
    },
}
PARAMS = PARAMS_BY_TRANSFORM[TARGET_TRANSFORM]

# ── 디바이스 ──
# models.DEVICE = 'gpu'

print(f'EXP: {EXP_ID} | USER: {USER}')
print(f'MODEL_NAME: {MODEL_NAME}')
print(f'N_TRIALS={N_TRIALS}, N_FOLDS={N_FOLDS}')
print(f'TARGET_TRANSFORM={TARGET_TRANSFORM} | CLIP_Y_EXTREME={CLIP_Y_EXTREME}')
print(f'REUSE_BEST_PARAMS={REUSE_BEST_PARAMS}')
print(f'OUT_DIR={OUT_DIR}')
print(f'DEVICE={models.DEVICE}')

EXP: reg-enet-001 | USER: jh
MODEL_NAME: enet
N_TRIALS=5, N_FOLDS=5
TARGET_TRANSFORM=log1p | CLIP_Y_EXTREME=True
REUSE_BEST_PARAMS=False
OUT_DIR=c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\reg_only\enet
DEVICE=cpu


## 3. 데이터 로드 + target clip + target transform 설정

In [3]:
# ── 데이터 로드 ──
xs, ys = load_all()
feat_cols = get_feat_cols(xs)
xs_dict = split_xs(xs)
print(f'xs: {xs.shape}, feat_cols: {len(feat_cols)}')

# ── y_train 극단값 clip ──
ys_input = {k: v.copy() for k, v in ys.items()}
if CLIP_Y_EXTREME:
    y_raw = ys_input['train'][TARGET_COL]
    second_max = y_raw[y_raw < y_raw.max()].max()
    n_clipped = (y_raw >= 1.0).sum()
    ys_input['train'][TARGET_COL] = y_raw.clip(upper=second_max)
    print(f'[CLIP_Y_EXTREME] 1.0 → {second_max:.6f} clip, {n_clipped}개')

# ── Target transform 함수 구성 ──
# 모델 학습은 transformed space, 최종 평가/저장은 original space
target_transform_fn = None
target_inverse_fn   = None

if TARGET_TRANSFORM == 'log1p':
    target_transform_fn = lambda y: np.log1p(np.asarray(y))
    target_inverse_fn   = lambda y: np.clip(np.expm1(np.asarray(y)), 0.0, None)
    print('[target transform] log1p / expm1(clip>=0)')

elif TARGET_TRANSFORM == 'yeo-johnson':
    from sklearn.preprocessing import PowerTransformer
    _pt = PowerTransformer(method='yeo-johnson', standardize=False)
    _y_clipped = ys_input['train'][TARGET_COL].values.reshape(-1, 1)
    _pt.fit(_y_clipped)
    _y_trans = _pt.transform(_y_clipped).ravel()
    _tt_min, _tt_max = float(_y_trans.min()), float(_y_trans.max())
    def target_transform_fn(y):
        return _pt.transform(np.asarray(y).reshape(-1, 1)).ravel()
    def target_inverse_fn(y):
        y_c = np.clip(np.asarray(y), _tt_min, _tt_max)
        out = _pt.inverse_transform(y_c.reshape(-1, 1)).ravel()
        return np.nan_to_num(np.clip(out, 0.0, None), nan=0.0, posinf=0.0, neginf=0.0)
    print(f'[target transform] yeo-johnson (범위 [{_tt_min:.4f}, {_tt_max:.4f}])')

elif TARGET_TRANSFORM == 'none':
    print('[target transform] none (원본 그대로)')

else:
    raise ValueError(f'Unknown TARGET_TRANSFORM: {TARGET_TRANSFORM!r}')

# ── 전처리 (고정 파이프라인) ──
pp = preprocess.run(xs, ys_input, feat_cols, xs_dict, params=PARAMS)
xs_train = pp['xs_train']
xs_val   = pp['xs_val']
xs_test  = pp['xs_test']
feat_cols_clean = pp['feat_cols']
print(f'\n[전처리 완료] feat_cols: {len(feat_cols_clean)}')

[load_xs] all-NaN 행 407개 제거 → 174,573행
[load_xs] 4 position 미만 unit 1개 제거 (split별: {'train': 1}) → die 174,573 → 174,572
[load_ys] train: xs에 없는 unit 60개 제거 → 26,187
[load_ys] validation: xs에 없는 unit 22개 제거 → 8,727
[load_ys] test: xs에 없는 unit 20개 제거 → 8,729
Xs: (174572, 1091)  |  Ys: train=26,187, val=8,727, test=8,729
xs: (174572, 1091), feat_cols: 1087
[CLIP_Y_EXTREME] 1.0 → 0.097417 clip, 1개
[target transform] log1p / expm1(clip>=0)
[Stage 0] 웨이퍼맵 사전 제외: 1087 → 1033 (54개 제거)
클리닝 파이프라인 시작
원본 feature 수: 1033
[상수/극저분산 제거] threshold=1e-06
  제거: 105개, 잔여: 928개
    컬럼: 1033 → 928 (105개 제거)
    DataFrame: (104748, 986)

[고결측 제거] threshold=50%
  제거: 5개, 잔여: 923개
    컬럼: 928 → 923 (5개 제거)
    DataFrame: (104748, 981)

[중복 컬럼 제거] sample_n=5000
  제거: 27개, 잔여: 896개
    컬럼: 923 → 896 (27개 제거)
    DataFrame: (104748, 954)

[고상관 제거] threshold=0.9, keep_by=std (std)
  제거: 332개, 잔여: 564개
    컬럼: 896 → 564 (332개 제거)
    DataFrame: (104748, 622)

[결측 indicator] 4개 컬럼 추가 (결측률 >= 25%)
[공간 보간 imputation]

## 4. Optuna HPO

In [ ]:
import json

best_params_for_refit = None
already_resolved = False
study = None

study_meta_for_save = {
    'exp_id':            EXP_ID,
    'exp_memo':          EXP_MEMO,
    'user':              USER,
    'model_name':        MODEL_NAME,
    'target_transform':  TARGET_TRANSFORM,
    'clip_y_extreme':    CLIP_Y_EXTREME,
    'effective_pp_params': pp['effective_params'],
    'n_trials':          N_TRIALS,
    'n_folds':           N_FOLDS,
    'seed':              SEED,
    'reuse_best_params': REUSE_BEST_PARAMS,
}

if REUSE_BEST_PARAMS:
    # Colab: ID가 있으면 gdown으로 JSON 다운로드
    try:
        import google.colab
        if GDRIVE_PREV_PARAMS_ID and not PREV_BEST_PARAMS_PATH:
            _dl = '/content/prev_best_params.json'
            os.system(f'gdown {GDRIVE_PREV_PARAMS_ID} -O {_dl}')
            PREV_BEST_PARAMS_PATH = _dl
    except ImportError:
        pass
    assert PREV_BEST_PARAMS_PATH and os.path.exists(PREV_BEST_PARAMS_PATH), \
        f'PREV_BEST_PARAMS_PATH 파일 없음: {PREV_BEST_PARAMS_PATH}'
    with open(PREV_BEST_PARAMS_PATH, encoding='utf-8') as f:
        _meta = json.load(f)
    best_params_for_refit = _meta['best_params_resolved']
    already_resolved = True
    study_meta_for_save['prev_best_params_path'] = PREV_BEST_PARAMS_PATH
    print(f'[REUSE] HPO 스킵, best_params 로드: {PREV_BEST_PARAMS_PATH}')
    print(f'  model_name(prev) = {_meta.get("model_name")}')
    print(f'  keys: {list(best_params_for_refit)[:8]} ...')
else:
    res = hpo.run_hpo(
        xs_train=xs_train,
        ys_train_unit=ys_input['train'],
        feat_cols=feat_cols_clean,
        model_name=MODEL_NAME,
        n_trials=N_TRIALS,
        n_folds=N_FOLDS,
        target_transform_fn=target_transform_fn,
        target_inverse_fn=target_inverse_fn,
        study_name=EXP_ID,
        storage=f'sqlite:///{DB_PATH}',
        user_attrs=study_meta_for_save,
        # ★ trial별 holdout RMSE 기록 (DB trial_user_attributes)
        xs_val=xs_val,   ys_val_unit=ys_input['validation'],
        xs_test=xs_test, ys_test_unit=ys_input['test'],
    )
    study                 = res['study']
    best_params_for_refit = res['best_params']
    study_meta_for_save['hpo_best_value'] = float(res['best_value'])
    print(f'\n[HPO 완료] best OOF RMSE = {res["best_value"]:.6f}')
    print(f'best_params = {best_params_for_refit}')

[I 2026-04-25 15:03:45,809] A new study created in RDB with name: reg-enet-001


  0%|          | 0/5 [00:00<?, ?it/s]

## 5. Best trial 재학습 (K-fold OOF)

In [ ]:
final = hpo.refit_best(
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    ys_train_unit=ys_input['train'],
    feat_cols=feat_cols_clean,
    model_name=MODEL_NAME,
    best_params=best_params_for_refit,
    already_resolved=already_resolved,
    n_folds=N_FOLDS,
    target_transform_fn=target_transform_fn,
    target_inverse_fn=target_inverse_fn,
)

y_true = ys_input['train'].set_index(KEY_COL)[TARGET_COL]
oof_u  = final['oof_pred_unit'].set_index(KEY_COL)['pred'].loc[y_true.index]
oof_rmse = float(np.sqrt(np.mean((oof_u.values - y_true.values)**2)))
print(f'\n[Refit 완료] OOF unit RMSE = {oof_rmse:.6f} (original space)')
print(f'fold_models: {len(final["fold_models"])}개')

[refit fold 1/5] tr_units=20949, vl_units=5238
[refit fold 2/5] tr_units=20949, vl_units=5238
[refit fold 3/5] tr_units=20950, vl_units=5237
[refit fold 4/5] tr_units=20950, vl_units=5237
[refit fold 5/5] tr_units=20950, vl_units=5237

[Refit 완료] OOF unit RMSE = 0.005527 (original space)
fold_models: 5개


## 6. 아티팩트 저장

In [ ]:
# ── Postprocess 설정 (집계 6종 + zero_clip 튜닝, π threshold는 reg 전용 경로라 미사용) ──
POSTPROCESS_CONFIG = {
    'agg_methods':      ('mean', 'median', 'max', 'min', 'trimmed_mean', 'weighted'),
    'zero_clip_range':  (0.001, 0.015),
    'zero_clip_step':   0.001,
    'use_pi_threshold': False,
}

hpo.save_artifacts(
    refit_result=final,
    xs_train=xs_train, xs_val=xs_val, xs_test=xs_test,
    out_dir=OUT_DIR, exp_id=EXP_ID,
    feature_names=feat_cols_clean,
    extra_feature_name=None,
    y_train_unit=ys_input['train'],
    y_val_unit=ys_input['validation'],   # ★ val_die.csv / val_unit.csv 에 health
    y_test_unit=ys_input['test'],        # ★ test_die.csv / test_unit.csv 에 health
    postprocess_config=POSTPROCESS_CONFIG,
    study_meta=study_meta_for_save,
)

for f in sorted(os.listdir(OUT_DIR)):
    size_kb = os.path.getsize(os.path.join(OUT_DIR, f)) / 1024
    print(f'  {f:30s}  {size_kb:10,.1f} KB')

[Aggregation] RMSEs: {'mean': 0.005527, 'median': 0.005527, 'max': 0.005534, 'min': 0.005531, 'trimmed_mean': 0.005527, 'weighted': 0.005527}
[Aggregation] best=median (0.005527)
[zero_clip] best=0.0010 (0.005527)
[Postprocess] best_agg=median, pi_th=None, zero_clip=0.0010, train_rmse=0.005527
[save_artifacts] c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\reg_only\catboost 저장 완료 (fold_models.pkl + best_params.json + 6 CSV, unit=tuned)
  best_params.json                       9.6 KB
  fold_models.pkl                    4,531.1 KB
  oof_die.csv                        5,580.6 KB
  oof_unit.csv                         941.7 KB
  optuna_jh_reg-catboost-001.db        120.0 KB
  test_die.csv                       1,860.5 KB
  test_unit.csv                        313.9 KB
  val_die.csv                        1,861.0 KB
  val_unit.csv                         314.1 KB


## 7. 요약

In [ ]:
print(f'★ 경로 C (단일 회귀)')
print(f'  EXP_ID       : {EXP_ID}')
print(f'  모델         : {MODEL_NAME}')
print(f'  transform    : {TARGET_TRANSFORM}')
print(f'  REUSE 모드   : {REUSE_BEST_PARAMS}')
print(f'  OOF RMSE     : {oof_rmse:.6f}')
if not REUSE_BEST_PARAMS and study is not None:
    print(f'  HPO best     : {res["best_value"]:.6f}')
    print(f'  trials       : {len(study.trials)}')
print(f'  feat_cols    : {len(feat_cols_clean)}')
print(f'  저장 경로    : {OUT_DIR}')
print(f'\n다른 모델도 돌리려면 MODEL_NAME 바꿔서 재실행. 5개 전부 돌리면 04_blend.ipynb로.')

★ 경로 C (단일 회귀)
  EXP_ID       : reg-catboost-001
  모델         : catboost
  transform    : log1p
  REUSE 모드   : False
  OOF RMSE     : 0.005527
  HPO best     : 0.005527
  trials       : 5
  feat_cols    : 568
  저장 경로    : c:\Users\Dell5371\Desktop\기업연계프로젝트\4_output\final\reg_only\catboost

다른 모델도 돌리려면 MODEL_NAME 바꿔서 재실행. 5개 전부 돌리면 04_blend.ipynb로.
